# Train the Lin et al. 2025 reproduction on Colab

This notebook trains the learned methods (MADDPG and MAPPO) and saves everything to your Google Drive.

## WARNING: approval needed first

Some training settings (called A-23 in `docs/lin2025_spec.md` Section 10) are proposals. The repo owner must approve them in `configs/train_lin2025.yaml` before any real training run. Do not run the training cells until that is done.

## 1. Connect Google Drive

Checkpoints and logs are saved to Drive so they survive if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Get the code

This downloads the repository and installs it. The next cell sets `CODE_REF`, the version of the code to use (a branch, tag or commit).

- The frozen environment is the git tag `lin-env-v1`, but that tag does not contain the training code (`src/uavmec/agents/` and `scripts/train.py`).
- The training code is on `main` after the R3 commit, so `CODE_REF` is `"main"` by default.
- The exact commit that was checked out is printed by the clone cell, and `train.py` also stores it in each run's `manifest.json`, so every run can be traced to its code.

In [ ]:
CODE_REF = "main"  # branch, tag or commit to check out

In [ ]:
!git clone https://github.com/tra134/Multi-Priority-Resource-Allocation-in-UAV-Networks-research.git /content/repo
%cd /content/repo
!git checkout {CODE_REF}
!git rev-parse HEAD
!pip install -q -e .

## 3. The runs to train

The paper's Table 2 has 7 rows; 6 need training and the seventh (Local Computing) is a fixed baseline:

1. POS MADDPG with SIC
2. FOS MADDPG with SIC
3. POS MAPPO with SIC
4. POS MADDPG without SIC (noSIC)
5. FOS MADDPG without SIC
6. POS MAPPO without SIC

Each one is trained for seeds 0 to 4. Change `SEED` in the cell, run it, then change it again. `--resume` continues from the last checkpoint if Colab stops.

Table 2 runs are saved under `MyDrive/uavmec/table2/`. The A-39 pilot runs live directly in `MyDrive/uavmec/` and are never reused for Table 2 (HUMAN decision, TE-025), so the two sets of folders must stay separate; otherwise `--resume` would continue a pilot run.

In [ ]:
SEED = 0  # change to 1, 2, 3, 4 for the other seeds

### POS MADDPG, SIC

In [ ]:
!python scripts/train.py --algo maddpg --mode POS --seed {SEED} --out /content/drive/MyDrive/uavmec/table2/pos_maddpg_sic_seed{SEED} --resume

### FOS MADDPG, SIC

In [ ]:
!python scripts/train.py --algo maddpg --mode FOS --seed {SEED} --out /content/drive/MyDrive/uavmec/table2/fos_maddpg_sic_seed{SEED} --resume

### POS MAPPO, SIC

In [ ]:
!python scripts/train.py --algo mappo --mode POS --seed {SEED} --out /content/drive/MyDrive/uavmec/table2/pos_mappo_sic_seed{SEED} --resume

### POS MADDPG, noSIC

In [ ]:
!python scripts/train.py --algo maddpg --mode POS --no-sic --seed {SEED} --out /content/drive/MyDrive/uavmec/table2/pos_maddpg_nosic_seed{SEED} --resume

### FOS MADDPG, noSIC

In [ ]:
!python scripts/train.py --algo maddpg --mode FOS --no-sic --seed {SEED} --out /content/drive/MyDrive/uavmec/table2/fos_maddpg_nosic_seed{SEED} --resume

### POS MAPPO, noSIC

In [ ]:
!python scripts/train.py --algo mappo --mode POS --no-sic --seed {SEED} --out /content/drive/MyDrive/uavmec/table2/pos_mappo_nosic_seed{SEED} --resume

## 4. Look at the results

Each run folder holds `train_log.csv` (one row per training episode), `eval_log.csv` (greedy evaluation on validation seeds), `manifest.json` (settings and code version), `ckpt.pt` and `final.pt`.